In [ ]:
import numpy as np
import torch
import time
from chronos import ChronosPipeline
import glob
from pathlib import Path
from datetime import datetime

In [ ]:
def load_multivariate_npy_dataset(file_path, index, split="test", batch_size=200):
    """
    Load and preprocess dataset from .npy file with dynamic batch handling
    """
    data = np.load(file_path, allow_pickle=True).item()

    split_length = len(data[split]["X"])
    max_index = min(index + batch_size, split_length)

    X_data = data[split]["X"][index:max_index]
    X_data = X_data.reshape(X_data.shape[0], -1)
    y_data = np.array([int(x) for x in data[split]["y"][index:max_index]])

    return X_data, y_data


def process_dataset(dataset_path, split="test", batch_size=200, output_prefix=None):
    """
    Process dataset and generate embeddings with dynamic naming and split selection
    """
    # Start timing
    start_time = time.time()

    initial_data = np.load(dataset_path, allow_pickle=True).item()
    data_len = len(initial_data[split]["X"])

    dataset_name = Path(dataset_path).stem
    if output_prefix is None:
        output_prefix = f"embedded_{dataset_name}_{split}"

    # Create dataset-specific output directory
    output_dir = Path(dataset_name)
    output_dir.mkdir(exist_ok=True)

    # Initialize timing statistics
    batch_times = []

    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map="mps" if torch.backends.mps.is_available() else "cpu",
        torch_dtype=torch.float16
        if torch.backends.mps.is_available()
        else torch.bfloat16,
    )

    batch_count = 0
    for i in range(0, data_len, batch_size):
        batch_start_time = time.time()
        batch_count += 1

        X_data, y_data = load_multivariate_npy_dataset(
            dataset_path, i, split, batch_size
        )

        context = torch.tensor(X_data)
        embeddings, tokenizer_state = pipeline.embed(context)

        save_path = output_dir / f"{output_prefix}_batch_{batch_count}.npy"
        with open(save_path, "wb") as f:
            np.save(f, embeddings.float().numpy())

        # Calculate and store batch processing time
        batch_time = time.time() - batch_start_time
        batch_times.append(batch_time)

        print(
            f"Processed batch {batch_count}/{(data_len + batch_size - 1) // batch_size} "
            f"(Time: {batch_time:.2f}s)"
        )

    # Calculate timing statistics
    total_time = time.time() - start_time
    avg_batch_time = np.mean(batch_times)

    print(f"\nProcessing Statistics for {dataset_name} {split} split:")
    print(f"Total batches: {batch_count}")
    print(f"Average time per batch: {avg_batch_time:.2f}s")
    print(f"Total processing time: {total_time:.2f}s")

    # Save timing information
    timing_info = {
        "dataset": dataset_name,
        "split": split,
        "total_batches": batch_count,
        "avg_batch_time": avg_batch_time,
        "total_time": total_time,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    timing_file = output_dir / f"{output_prefix}_timing_info.npy"
    np.save(timing_file, timing_info)

    return output_prefix, batch_count


def concatenate_embeddings(prefix, dataset_name, output_name=None):
    """
    Concatenate embeddings from multiple files with dynamic naming
    """
    file_pattern = f"{dataset_name}/{prefix}_batch_*.npy"
    file_list = sorted(glob.glob(file_pattern))

    if not file_list:
        raise ValueError(f"No files found matching pattern: {file_pattern}")

    print(f"Found {len(file_list)} files to concatenate")

    start_time = time.time()

    all_embeddings = []
    for file_path in file_list:
        embeddings = np.load(file_path)
        all_embeddings.append(embeddings)
        print(f"Loaded: {Path(file_path).name}")

    concatenated_embeddings = np.concatenate(all_embeddings, axis=0)

    if output_name is None:
        output_name = f"{prefix}_concatenated.npy"

    output_path = Path(dataset_name) / output_name
    np.save(
        output_path,
        concatenated_embeddings.reshape(concatenated_embeddings.shape[0], -1),
    )

    concat_time = time.time() - start_time

    print(f"\nConcatenation Statistics:")
    print(f"Concatenated embeddings shape: {concatenated_embeddings.shape}")
    print(f"Concatenation time: {concat_time:.2f}s")
    print(f"Embeddings saved to: {output_path}")

    return concatenated_embeddings


def process_and_concatenate(
    dataset_path, split="test", batch_size=200, output_prefix=None, final_name=None
):
    """
    Complete pipeline: process dataset and concatenate results with split selection
    """
    dataset_name = Path(dataset_path).stem
    prefix, _ = process_dataset(dataset_path, split, batch_size, output_prefix)
    return concatenate_embeddings(prefix, dataset_name, final_name)

In [ ]:
dataset_path = "MP8.npy"
split = "train"
output_prefix = "MP8_train"
final_name = "MP8_train_embeddings.npy"

# Run the complete pipeline
final_embeddings = process_and_concatenate(
    dataset_path=dataset_path,
    split=split,
    batch_size=200,
    output_prefix=output_prefix,
    final_name=final_name,
)

split = "test"
output_prefix = "MP8_test"
final_name = "MP8_test_embeddings.npy"

# Run the complete pipeline
final_embeddings = process_and_concatenate(
    dataset_path=dataset_path,
    split=split,
    batch_size=200,
    output_prefix=output_prefix,
    final_name=final_name,
)

In [ ]:
train_data = np.load("MP8/MP8_train_embeddings.npy", allow_pickle=True)
test_data = np.load("MP8/MP8_test_embeddings.npy", allow_pickle=True)

In [ ]:
def load_npy_dataset(file_path):
    data = np.load(file_path, allow_pickle=True).item()
    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])
    return X_train, y_train, X_test, y_test


_, y_train, _, y_test = load_npy_dataset("MP8.npy")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import os

os.environ["MallocStackLogging"] = "1"

# Initialize the classifier
classifier = RandomForestClassifier()

# Measure training time
fit_start_time = time.time()
classifier.fit(train_data, y_train)
fit_end_time = time.time()
fit_time = fit_end_time - fit_start_time

# Measure prediction time
pred_start_time = time.time()
y_pred = classifier.predict(test_data)
pred_end_time = time.time()
pred_time = pred_end_time - pred_start_time

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)

# Print results
print("Random Forest Classification Results")
print("-" * 35)
print(f"Training Time: {fit_time:.4f} seconds")
print(f"Prediction Time: {pred_time:.4f} seconds")
print(f"Model Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize the classifier
classifier = LogisticRegression()

# Measure training time
fit_start_time = time.time()
classifier.fit(train_data, y_train)
fit_end_time = time.time()
fit_time = fit_end_time - fit_start_time

# Measure prediction time
pred_start_time = time.time()
y_pred = classifier.predict(test_data)
pred_end_time = time.time()
pred_time = pred_end_time - pred_start_time

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
# Print results
print("Logistic Classification Results")
print("-" * 35)
print(f"Training Time: {fit_time:.4f} seconds")
print(f"Prediction Time: {pred_time:.4f} seconds")
print(f"Model Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.linear_model import RidgeClassifier

# Initialize the classifier
classifier = RidgeClassifier()

# Measure training time
fit_start_time = time.time()
classifier.fit(train_data, y_train)
fit_end_time = time.time()
fit_time = fit_end_time - fit_start_time

# Measure prediction time
pred_start_time = time.time()
y_pred = classifier.predict(test_data)
pred_end_time = time.time()
pred_time = pred_end_time - pred_start_time

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)

# Print results
print("Ridge Classification Results")
print("-" * 35)
print(f"Training Time: {fit_time:.4f} seconds")
print(f"Prediction Time: {pred_time:.4f} seconds")
print(f"Model Accuracy: {accuracy:.4f}")